# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display key metadata fields
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")


## 2. Data Overview
Review available record sets and their `@id`, as well as fields and columns with their `@id`s.

Croissant datasets define their structure in terms of record sets, fields, and columns. We'll enumerate these using the dataset's metadata.

In [ ]:
# Gather and display all record sets and their corresponding field and column @id's

record_sets = list(dataset.iter_record_sets())
print(f"Number of record sets: {len(record_sets)}\n")

record_set_ids = []
record_set_fields = {}  # record_set_id -> list of field @ids

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    print(f"Record Set: {rs_id}")
    fields = list(rs.iter_fields())
    field_ids = []
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_ids.append(field_id)
        print(f"  Field: {field_id} -- DataType: {getattr(field, 'dataType', None)} Name: {getattr(field, 'name', None)}")
        columns = getattr(field, 'columns', None)
        if columns:
            for column in columns:
                col_id = getattr(column, '@id', None)
                print(f"    Column: {col_id} Name: {getattr(column, 'name', None)}")
    record_set_fields[rs_id] = field_ids
    print()

# For later use, let's also build a dictionary from record set id to field id list
# and display for user reference
print("\nAll record set @ids:")
for idx, rs_id in enumerate(record_set_ids):
    print(f"  {idx}: {rs_id}")


## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

We'll extract all available record sets for further exploration.

In [ ]:
# Load data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records for record set {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} ({len(records)} records)")
    print("Fields:", df.columns.tolist())

# For demonstration, pick the first non-empty record set
main_record_set_id = next(iter(dataframes.keys()))
print(f"\nDisplaying the first 5 rows of the main record set ({main_record_set_id}):\n")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate how to filter, normalize, and group data. We'll select a numeric field from our main record set. Insert the appropriate `@id` for a numeric or score field as identified above.

In [ ]:
# --- Adjust the following as needed for your dataset ---
df = dataframes[main_record_set_id]

# Choose a field (column) ID for a numeric field, e.g., PHQ-9 score, GAD-7, or PSQ
numeric_field = None
for col in df.columns:
    # Try to find a likely numeric field
    if any(token in col.lower() for token in ["score", "phq", "gad", "psq", "total"]):
        numeric_field = col
        break
# Fallback: pick the first numeric typed column
if numeric_field is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

assert numeric_field is not None, "Could not find a suitable numeric field in the data."

print(f"Using numeric field: {numeric_field}")
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Now attempt to group by a categorical field, e.g. 'gender', 'village', or similar
group_field = None
for col in df.columns:
    if any(x in col.lower() for x in ['gender', 'village', 'education', 'marital']):
        group_field = col
        break
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data (mean) by {group_field}:")
    display(grouped_df[[numeric_field, f"{numeric_field}_normalized"]] if f"{numeric_field}_normalized" in grouped_df else grouped_df)
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], kde=True, bins=15)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot of the numeric field grouped by group_field, if available:
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f'{numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey dataset using the `mlcroissant` library. We investigated the record set structure, extracted data, and performed basic filtering, normalization, and grouping on a selected numeric field. Visualizations revealed the distribution and potential group differences in key mental health indicators.

For deeper domain-specific analysis, consult the field metadata for meaning and examine the detailed Croissant schema at the source URL. This dataset enables further studies in mental health data science for the Kilifi County population.